# Reduce LLM Costs With Semantic Caching

[Read the cookbook page](https://docs.futureagi.com/docs/cookbook/command-center/semantic-caching)

Ask a Ridgeline Outfitters FAQ twice, then change its wording. The `x-agentcc-cache` header distinguishes an exact hit, a similarity hit, and a new provider call. Similar wording can qualify for a hit; similar intent alone is not enough.

## Install

Start with a fresh Python 3.12 runtime. Set the named keys in your environment or Colab Secrets, or enter them at the hidden prompts.

In [ ]:
%pip install openai

In [ ]:
import os
from getpass import getpass

def load_secret(name):
    if os.environ.get(name):
        return
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except (ImportError, KeyError, RuntimeError):
        value = None
    except Exception as exc:
        if type(exc).__name__ not in {"SecretNotFoundError", "NotebookAccessError"}:
            raise
        value = None
    value = value or getpass(f"{name}: ")
    if not value:
        raise ValueError(f"{name} is required")
    os.environ[name] = value

for name in ['AGENTCC_API_KEY']:
    load_secret(name)

## Send a baseline request and note the cost

Use a development gateway with a provider configured and both L1 and L2 caches available. For self-hosting, set `AGENTCC_BASE_URL` to its OpenAI-compatible `/v1` URL.

In [ ]:
import os
from uuid import uuid4
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["AGENTCC_API_KEY"],
    base_url=os.environ.get("AGENTCC_BASE_URL", "https://gateway.futureagi.com/v1"),
    default_headers={"x-agentcc-cache-namespace": f"returns-{uuid4().hex}"},
)
policy = (
    "You help Ridgeline Outfitters shoppers. Unused gear can be returned within "
    "30 days. A prepaid label is included. Answer in one sentence; do not invent details."
)
question = "How long do I have to return unused gear to Ridgeline Outfitters?"

def ask(text, headers=None):
    return client.chat.completions.with_raw_response.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": policy}, {"role": "user", "content": text}],
        extra_headers=headers or {},
    )

baseline = ask(question, {"Cache-Control": "no-store"})
print("baseline:", baseline.headers.get("x-agentcc-cache", "not reported"))
print("cost (USD):", baseline.headers.get("x-agentcc-cost", "not reported"))
print("latency (ms):", baseline.headers.get("x-agentcc-latency-ms", "not reported"))

## Turn on exact caching in the dashboard

Enable Response Cache under **Gateway → Providers → Cache**. Use a TTL of one hour.

In [ ]:
r1 = ask(question)
r2 = ask(question)
print("call 1:", r1.headers.get("x-agentcc-cache", "not reported"))
print("call 2:", r2.headers.get("x-agentcc-cache", "not reported"))
print("same completion:", r1.parse().id == r2.parse().id)

## Switch to semantic caching for paraphrased prompts

Enable L2 Semantic Cache and set its similarity threshold to `0.92`. For self-hosting, enable and configure the semantic store at gateway startup as well.

In [ ]:
anchor = ask(question, {"x-agentcc-cache-force-refresh": "true"})
near = ask("How long do I have to return unused gear to Ridgeline Outfitters, please?")
other = ask("Is a prepaid shipping label included?")
for label, reply in [("anchor", anchor), ("near match", near), ("shipping label", other)]:
    print(label, reply.headers.get("x-agentcc-cache", "not reported"))
print("near match reused completion:", near.parse().id == anchor.parse().id)

## Measure the savings

Count the headers for 15 requests. An absent cost header is not zero cost.

In [ ]:
from collections import Counter
from decimal import Decimal
from time import perf_counter

batch = [question, "How long do I have to return unused gear to Ridgeline Outfitters, please?",
         "Is a prepaid shipping label included?"] * 5
tally = Counter()
costs = []
start = perf_counter()
for text in batch:
    headers = ask(text).headers
    tally[headers.get("x-agentcc-cache", "not reported")] += 1
    if "x-agentcc-cost" in headers:
        costs.append(Decimal(headers["x-agentcc-cost"]))
print("requests:", sum(tally.values()), "cache results:", dict(tally))
print("total reported cost (USD):", sum(costs) if len(costs) == len(batch) else "incomplete")
print(f"wall time: {perf_counter() - start:.1f}s")

## Bypass or invalidate the cache when you need fresh answers

Expect a miss followed by an exact hit with the refreshed completion ID.

In [ ]:
fresh = ask(question, {"x-agentcc-cache-force-refresh": "true"})
repeat = ask(question)
print("forced refresh:", fresh.headers.get("x-agentcc-cache", "not reported"))
print("after refresh:", repeat.headers.get("x-agentcc-cache", "not reported"))
print("refreshed completion reused:", repeat.parse().id == fresh.parse().id)